# Vous trouverez dans les scripts les logos obtenus via Gemini, son utilisation à été une aide mais ils sont surtout présent pour aider la lisibilité des cellules de rendu (on si perd trop facilement sinon 😭 ).
Signé Raphaël PHAN👋


In [2]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')
path_racine = '/content/drive/MyDrive/YBOOST DATA B2/csv/'

Mounted at /content/drive


# **Importation des datasets**

## NB - Import des datasets des nombres de validations depuis 2015 jusqu'à 2024

In [3]:
all_nb_data = []
total_files = 0
correct_files = 0
failed_files = 0

print("=== 🛰️ DÉMARRAGE DU SCANNER DE DONNÉES NB (RER A) ===")

for root, dirs, files in os.walk(path_racine):
    print(f"\n📂 --- Entrée dans le dossier : {os.path.basename(root)} ---")

    for file in files:
        if "NB" in file and "FER" in file and (file.endswith('.csv') or file.endswith('.txt')):
            total_files += 1
            file_path = os.path.join(root, file)
            print(f"   🔍 Vérification de : {file}")

            df_full = None
            try:
                # TENTATIVE 1 : Latin1
                df_full = pd.read_csv(file_path, sep=None, engine='python', encoding='latin1')

                # On retire les caractères invisibles (BOM) et on met tout en MAJUSCULES
                df_full.columns = [c.strip().replace('ï»¿', '').upper() for c in df_full.columns]

                if 'CODE_STIF_RES' not in df_full.columns:
                    raise ValueError("Structure non reconnue")

            except:
                try:
                    # TENTATIVE 2 : UTF-16 (Secours)
                    df_full = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-16')
                    df_full.columns = [c.strip().replace('ï»¿', '').upper() for c in df_full.columns]
                    print("     💡 Note : Encodage UTF-16 détecté.")
                except Exception as e:
                    print(f"     💥 ÉCHEC CRITIQUE sur {file} : {e}")
                    failed_files += 1
                    continue

            # FILTRAGE ET VALIDATION
            if df_full is not None and 'CODE_STIF_RES' in df_full.columns:
                df_full['CODE_STIF_RES'] = pd.to_numeric(df_full['CODE_STIF_RES'], errors='coerce')
                df_rera = df_full[df_full['CODE_STIF_RES'] == 801].copy()

                if not df_rera.empty:
                    # On s'assure que la colonne de date s'appelle bien 'JOUR'
                    # (Gestion des variantes comme 'Jour' ou 'JOUR ')
                    if 'JOUR' in df_rera.columns:
                        all_nb_data.append(df_rera)
                        correct_files += 1
                        print(f"     ✅ SUCCÈS : {len(df_rera)} lignes ajoutées.")
                    else:
                        print(f"     ❌ ERREUR : Colonne de date introuvable dans {file}")
                        failed_files += 1
                else:
                    print(f"     ⚠️ VIDE : Aucune ligne '801' trouvée.")
                    failed_files += 1

if all_nb_data:
    df_nb_base = pd.concat(all_nb_data, ignore_index=True)
    print(f"\n" + "="*50)
    print(f"=== 📊 RÉSUMÉ FINAL DU TRAITEMENT ===")
    print(f"📁 Total de fichiers examinés : {total_files}")
    print(f"✅ Fichiers importés avec succès : {correct_files}")
    print(f"❌ Fichiers en échec ou vides : {failed_files}")
    print(f"📈 Total de lignes NB fusionnées pour le RER A : {len(df_nb_base)}")
    print("="*50)

=== 🛰️ DÉMARRAGE DU SCANNER DE DONNÉES NB (RER A) ===

📂 --- Entrée dans le dossier :  ---

📂 --- Entrée dans le dossier : data-rf-2024 ---

📂 --- Entrée dans le dossier : data-rf-2024 ---
   🔍 Vérification de : 2024_S1_NB_FER.txt
     ✅ SUCCÈS : 51103 lignes ajoutées.
   🔍 Vérification de : 2024_T3_NB_FER.txt
     ✅ SUCCÈS : 29258 lignes ajoutées.
   🔍 Vérification de : 2024_T4_NB_FER.txt
     ✅ SUCCÈS : 29360 lignes ajoutées.

📂 --- Entrée dans le dossier : data-rf-2018 ---

📂 --- Entrée dans le dossier : data-rf-2018 ---
   🔍 Vérification de : 2018_S1_NB_FER.txt
     ✅ SUCCÈS : 60595 lignes ajoutées.

📂 --- Entrée dans le dossier : data-rf-2022 ---

📂 --- Entrée dans le dossier : data-rf-2022 ---
   🔍 Vérification de : 2022_S2_NB_FER.txt
     ✅ SUCCÈS : 68254 lignes ajoutées.
   🔍 Vérification de : 2022_S1_NB_FER.txt
     ✅ SUCCÈS : 69312 lignes ajoutées.

📂 --- Entrée dans le dossier : data-rf-2019 ---

📂 --- Entrée dans le dossier : data-rf-2019 ---
   🔍 Vérification de : 2019_S1_

In [4]:
df_nb_base.head()

,JOUR,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_ZDC,CATEGORIE_TITRE,NB_VALD,ID_REFA_LDA,LDA
0,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Amethyste,29,NaN,NaN
1,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Contrat Solidarité Transport,170,NaN,NaN
2,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Forfait Navigo,619,NaN,NaN
3,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Forfaits courts,3,NaN,NaN
4,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Imagine R,309,NaN,NaN


In [5]:
print(len(df_nb_base))

1129064


## PROFIL - Import des datasets des profils horaires depuis 2015 jusqu'à 2024

In [6]:
all_profil_data = []

total_files = 0
correct_files = 0
failed_files = 0

print("=== 🛰️ DÉMARRAGE DU SCANNER DE PROFILS (RER A) ===")

for root, dirs, files in os.walk(path_racine):
    print(f"\n📂 --- Entrée dans le dossier : {os.path.basename(root)} ---")

    for file in files:
        if "PROFIL" in file and "FER" in file and (file.endswith('.csv') or file.endswith('.txt')):
            total_files += 1
            file_path = os.path.join(root, file)
            print(f"  🔍 Vérification de : {file}")

            # --- AJOUT : Extraction de la période depuis le nom du fichier ---
            periode_extraite = file.split('_PROFIL')[0]

            df_full = None
            # TENTATIVE 1 : Latin1
            try:
                df_full = pd.read_csv(file_path, sep=None, engine='python', encoding='latin1', dtype=str)
                df_full.columns = [c.strip() for c in df_full.columns]

                if 'CODE_STIF_RES' not in df_full.columns:
                    raise ValueError("Structure non reconnue")
            except:
                # TENTATIVE 2 : UTF-16 (Secours pour 2023_S2)
                try:
                    df_full = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-16', dtype=str)
                    df_full.columns = [c.strip() for c in df_full.columns]
                    print("     💡 Note : Encodage UTF-16 détecté.")
                except Exception as e:
                    print(f"     💥 ÉCHEC CRITIQUE sur {file} : {e}")
                    failed_files += 1
                    continue

            # FILTRAGE RER A (Code 801)
            if df_full is not None and 'CODE_STIF_RES' in df_full.columns:
                # Conversion propre en numérique pour le filtre
                df_full['CODE_STIF_RES_NUM'] = pd.to_numeric(df_full['CODE_STIF_RES'], errors='coerce')
                df_profil_rera = df_full[df_full['CODE_STIF_RES_NUM'] == 801].copy()

                if not df_profil_rera.empty:
                    # --- AJOUT : Création de la colonne PERIODE ---
                    df_profil_rera['PERIODE'] = periode_extraite

                    all_profil_data.append(df_profil_rera)
                    correct_files += 1
                    print(f"     ✅ SUCCÈS : Données horaires RER A ajoutées ({periode_extraite}).")
                else:
                    print(f"     ⚠️ VIDE : Aucune donnée pour la ligne A (801).")
                    failed_files += 1
            else:
                print(f"     ❌ ERREUR : Colonne 'CODE_STIF_RES' introuvable.")
                failed_files += 1

if all_profil_data:
    df_profil_base = pd.concat(all_profil_data, ignore_index=True)
    print(f"\n" + "="*50)
    print(f"=== 📊 RÉSUMÉ FINAL DU TRAITEMENT (PROFIL) ===")
    print(f"📁 Total de fichiers examinés : {total_files}")
    print(f"✅ Fichiers importés avec succès : {correct_files}")
    print(f"❌ Fichiers en échec ou vides : {failed_files}")
    print(f"📈 Total de lignes PROFIL fusionnées pour le RER A : {len(df_profil_base)}")
    print("="*50)
else:
    print("\n❌ ERREUR : Aucun profil n'a pu être chargé.")

=== 🛰️ DÉMARRAGE DU SCANNER DE PROFILS (RER A) ===

📂 --- Entrée dans le dossier :  ---

📂 --- Entrée dans le dossier : data-rf-2024 ---

📂 --- Entrée dans le dossier : data-rf-2024 ---
  🔍 Vérification de : 2024_S1_PROFIL_FER.txt
     ✅ SUCCÈS : Données horaires RER A ajoutées (2024_S1).
  🔍 Vérification de : 2024_T3_PROFIL_FER.txt
     ✅ SUCCÈS : Données horaires RER A ajoutées (2024_T3).
  🔍 Vérification de : 2024_T4_PROFIL_FER.txt
     ✅ SUCCÈS : Données horaires RER A ajoutées (2024_T4).

📂 --- Entrée dans le dossier : data-rf-2018 ---

📂 --- Entrée dans le dossier : data-rf-2018 ---
  🔍 Vérification de : 2018_S1_PROFIL_FER.txt
     ✅ SUCCÈS : Données horaires RER A ajoutées (2018_S1).

📂 --- Entrée dans le dossier : data-rf-2022 ---

📂 --- Entrée dans le dossier : data-rf-2022 ---
  🔍 Vérification de : 2022_S1_PROFIL_FER.txt
     ✅ SUCCÈS : Données horaires RER A ajoutées (2022_S1).
  🔍 Vérification de : 2022_S2_PROFIL_FER.txt
     ✅ SUCCÈS : Données horaires RER A ajoutées (2022

In [7]:
df_profil_base.head()

,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_ZDC,CAT_JOUR,TRNC_HORR_60,pourc_validations,CODE_STIF_RES_NUM,PERIODE,Pourcentage_validations,ID_REFA_LDA,ï»¿CODE_STIF_TRNS,lda
0,810,801,116,BRY-SUR-MARNE,73166,DIJFP,0H-1H,"0,35",801.0,2024_S1,NaN,NaN,NaN,NaN
1,810,801,116,BRY-SUR-MARNE,73166,DIJFP,10H-11H,"7,21",801.0,2024_S1,NaN,NaN,NaN,NaN
2,810,801,116,BRY-SUR-MARNE,73166,DIJFP,11H-12H,"7,48",801.0,2024_S1,NaN,NaN,NaN,NaN
3,810,801,116,BRY-SUR-MARNE,73166,DIJFP,12H-13H,"7,4",801.0,2024_S1,NaN,NaN,NaN,NaN
4,810,801,116,BRY-SUR-MARNE,73166,DIJFP,13H-14H,"8,16",801.0,2024_S1,NaN,NaN,NaN,NaN


### Persistance des datasets fusionnés

In [8]:
#Récupération des csv
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde
df_nb_base.to_csv(os.path.join(output_base_path, 'RER_A_NB_Base.csv'), index=False)
df_profil_base.to_csv(os.path.join(output_base_path, 'RER_A_PROFIL_Base.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


# Netttoyage

In [9]:
#Récupération des csv sauvegardés
df_nb = pd.read_csv(os.path.join(output_base_path, 'RER_A_NB_Base.csv'))
df_profil = pd.read_csv(os.path.join(output_base_path, 'RER_A_PROFIL_Base.csv'))
print("Datasets chargés avec succès !")

/tmp/ipykernel_1246/2746436933.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nb = pd.read_csv(os.path.join(output_base_path, 'RER_A_NB_Base.csv'))


Datasets chargés avec succès !


/tmp/ipykernel_1246/2746436933.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_profil = pd.read_csv(os.path.join(output_base_path, 'RER_A_PROFIL_Base.csv'))


In [10]:
print(len(df_nb_base))
print(len(df_nb))
print(len(df_profil_base))
print(len(df_profil))

1129064
1129064
103614
103614


## NB

In [11]:
df_nb.head()

,JOUR,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_ZDC,CATEGORIE_TITRE,NB_VALD,ID_REFA_LDA,LDA
0,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Amethyste,29,NaN,NaN
1,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Contrat Solidarité Transport,170,NaN,NaN
2,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Forfait Navigo,619,NaN,NaN
3,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Forfaits courts,3,NaN,NaN
4,01/01/2024,810,801.0,116.0,BRY-SUR-MARNE,73166.0,Imagine R,309,NaN,NaN


In [12]:
# Correction du bug d'importation (caractères ï»¿) et suppression des espaces
df_nb.rename(columns={'ï»¿JOUR': 'JOUR', 'JOUR ': 'JOUR'}, inplace=True)
df_nb.columns = [c.strip() for c in df_nb.columns]
print("Colonnes nettoyées :", df_nb.columns.tolist())

Colonnes nettoyées : ['JOUR', 'CODE_STIF_TRNS', 'CODE_STIF_RES', 'CODE_STIF_ARRET', 'LIBELLE_ARRET', 'ID_ZDC', 'CATEGORIE_TITRE', 'NB_VALD', 'ID_REFA_LDA', 'LDA']


In [13]:
df_nb.isnull().sum()

,0
JOUR,0
CODE_STIF_TRNS,0
CODE_STIF_RES,0
CODE_STIF_ARRET,0
LIBELLE_ARRET,0
ID_ZDC,968042
CATEGORIE_TITRE,0
NB_VALD,0
ID_REFA_LDA,296668
LDA,993418


In [14]:
df_nb.dtypes

,0
JOUR,object
CODE_STIF_TRNS,int64
CODE_STIF_RES,float64
CODE_STIF_ARRET,float64
LIBELLE_ARRET,object
ID_ZDC,float64
CATEGORIE_TITRE,object
NB_VALD,object
ID_REFA_LDA,float64
LDA,float64


In [15]:
# Correction du bug d'importation (caractères ï»¿) et suppression des espaces
df_nb.rename(columns={'ï»¿JOUR': 'JOUR', 'JOUR ': 'JOUR'}, inplace=True)
df_nb.columns = [c.strip() for c in df_nb.columns]
print("Colonnes nettoyées :", df_nb.columns.tolist())

Colonnes nettoyées : ['JOUR', 'CODE_STIF_TRNS', 'CODE_STIF_RES', 'CODE_STIF_ARRET', 'LIBELLE_ARRET', 'ID_ZDC', 'CATEGORIE_TITRE', 'NB_VALD', 'ID_REFA_LDA', 'LDA']


In [16]:
print(len(df_nb))
print(len(df_profil))

1129064
103614


In [17]:
# Transformation du texte en objet Date
df_nb['JOUR'] = pd.to_datetime(df_nb['JOUR'], errors='coerce')
# On vérifie qu'il n'y a pas de dates absurdes
print("Plage temporelle :", df_nb['JOUR'].min(), "à", df_nb['JOUR'].max())

Plage temporelle : 2015-01-01 00:00:00 à 2024-12-06 00:00:00


In [18]:
# Conversion en numérique et remplacement des erreurs par 0
df_nb['NB_VALD'] = pd.to_numeric(df_nb['NB_VALD'], errors='coerce').fillna(0).astype(int)
print("Moyenne des validations par ligne :", df_nb['NB_VALD'].mean())

Moyenne des validations par ligne : 1731.4588951556334


In [19]:
# Suppression des lignes strictement identiques
avant = len(df_nb)
df_nb = df_nb.drop_duplicates()
print(f"Doublons supprimés : {avant - len(df_nb)} lignes.")

Doublons supprimés : 374919 lignes.


In [20]:
# On supprime les lignes où l'arrêt ou la date sont inconnus
df_nb = df_nb.dropna(subset=['JOUR', 'LIBELLE_ARRET'])
print("Valeurs nulles restantes :", df_nb.isnull().sum().sum())

Valeurs nulles restantes : 844048


In [21]:
print(len(df_nb))

422024


In [22]:
# Tri par date et remise à zéro de l'index (de 0 à 1 129 064)
df_nb = df_nb.sort_values(by='JOUR').reset_index(drop=True)
df_nb.head()

,JOUR,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_ZDC,CATEGORIE_TITRE,NB_VALD,ID_REFA_LDA,LDA
0,2015-01-01,810,801.0,6.0,ACHERES-VILLE,NaN,FGT,43,73604.0,NaN
1,2015-01-01,810,801.0,591.0,NEUVILLE UNIVERSITE,NaN,FGT,31,66436.0,NaN
2,2015-01-01,810,801.0,591.0,NEUVILLE UNIVERSITE,NaN,IMAGINE R,95,66436.0,NaN
3,2015-01-01,810,801.0,591.0,NEUVILLE UNIVERSITE,NaN,NAVIGO,111,66436.0,NaN
4,2015-01-01,810,801.0,591.0,NEUVILLE UNIVERSITE,NaN,TST,23,66436.0,NaN


In [23]:
df_nb.isnull().sum()


,0
JOUR,0
CODE_STIF_TRNS,0
CODE_STIF_RES,0
CODE_STIF_ARRET,0
LIBELLE_ARRET,0
ID_ZDC,381832
CATEGORIE_TITRE,0
NB_VALD,0
ID_REFA_LDA,93690
LDA,368526


In [24]:
# Colonnes à garder pour le volume de trafic (NB)
cols_nb = ['JOUR', 'LIBELLE_ARRET', 'CATEGORIE_TITRE', 'NB_VALD']
df_nb = df_nb[cols_nb].copy()

print("Datasets allégés ! Colonnes restantes :", df_nb.columns.tolist())

Datasets allégés ! Colonnes restantes : ['JOUR', 'LIBELLE_ARRET', 'CATEGORIE_TITRE', 'NB_VALD']


In [25]:
df_nb.head()

,JOUR,LIBELLE_ARRET,CATEGORIE_TITRE,NB_VALD
0,2015-01-01,ACHERES-VILLE,FGT,43
1,2015-01-01,NEUVILLE UNIVERSITE,FGT,31
2,2015-01-01,NEUVILLE UNIVERSITE,IMAGINE R,95
3,2015-01-01,NEUVILLE UNIVERSITE,NAVIGO,111
4,2015-01-01,NEUVILLE UNIVERSITE,TST,23


In [26]:
# 1. Conversion en vrai format Date (si ce n'est pas déjà fait)
df_nb['JOUR'] = pd.to_datetime(df_nb['JOUR'], errors='coerce')

# 2. Extraction du mois (influence saisonnière)
df_nb['MOIS'] = df_nb['JOUR'].dt.month

# 3. Extraction du jour de la semaine (0=Lundi, 6=Dimanche)
df_nb['JOUR_SEMAINE'] = df_nb['JOUR'].dt.dayofweek

# 4. Identification du weekend (variable binaire pour l'IA)
df_nb['EST_WEEKEND'] = df_nb['JOUR_SEMAINE'].isin([5, 6]).astype(int)

print("Variables temporelles créées pour le Machine Learning !")
df_nb.head()

Variables temporelles créées pour le Machine Learning !


,JOUR,LIBELLE_ARRET,CATEGORIE_TITRE,NB_VALD,MOIS,JOUR_SEMAINE,EST_WEEKEND
0,2015-01-01,ACHERES-VILLE,FGT,43,1,3,0
1,2015-01-01,NEUVILLE UNIVERSITE,FGT,31,1,3,0
2,2015-01-01,NEUVILLE UNIVERSITE,IMAGINE R,95,1,3,0
3,2015-01-01,NEUVILLE UNIVERSITE,NAVIGO,111,1,3,0
4,2015-01-01,NEUVILLE UNIVERSITE,TST,23,1,3,0


In [27]:
def determiner_cat_jour(date):
    if date.weekday() < 5:      # 0 à 4 = Lundi à Vendredi
        return 'JOHV'
    elif date.weekday() == 5:   # 5 = Samedi
        return 'SAHV'
    else:                       # 6 = Dimanche
        return 'DIJFP'

# Application de la fonction
df_nb['CAT_JOUR'] = df_nb['JOUR'].apply(determiner_cat_jour)
print("Colonnes de correspondance créées !")

Colonnes de correspondance créées !


In [28]:
# Tri du fichier des volumes journaliers
df_nb = df_nb.sort_values(
    by=['JOUR', 'LIBELLE_ARRET'],
    ascending=[True, True]
).reset_index(drop=True)

print("Tri du dataset NB terminé !")

Tri du dataset NB terminé !


In [29]:
# 1. Si la colonne CATEGORIE_TITRE est encore là, on la retire pour "forcer" la fusion
if 'CATEGORIE_TITRE' in df_nb.columns:
    df_nb_sans_titre = df_nb.drop(columns=['CATEGORIE_TITRE'])
else:
    df_nb_sans_titre = df_nb.copy()

# 2. On définit nos critères de regroupement (tout sauf le nombre de validations)
colonnes_groupement = [col for col in df_nb_sans_titre.columns if col != 'NB_VALD']

# 3. L'opération magique : On regroupe par Date/Gare et on ADDITIONNE les validations
df_nb_total = df_nb_sans_titre.groupby(colonnes_groupement, as_index=False)['NB_VALD'].sum()

# 4. Vérification
print(f"Lignes avec le détail des catégories : {len(df_nb)}")
print(f"Lignes avec le TOTAL GLOBAL par jour et par gare : {len(df_nb_total)}")

# On met à jour ton dataframe principal
df_nb = df_nb_total.copy()
df_nb.head(10)

Lignes avec le détail des catégories : 422024
Lignes avec le TOTAL GLOBAL par jour et par gare : 59203


,JOUR,LIBELLE_ARRET,MOIS,JOUR_SEMAINE,EST_WEEKEND,CAT_JOUR,NB_VALD
0,2015-01-01,ACHERES-GRAND-CORMIER,1,3,0,JOHV,8
1,2015-01-01,ACHERES-VILLE,1,3,0,JOHV,482
2,2015-01-01,AUBER,1,3,0,JOHV,1443
3,2015-01-01,BOISSY-SAINT-LEGER,1,3,0,JOHV,1228
4,2015-01-01,BRY-SUR-MARNE,1,3,0,JOHV,800
5,2015-01-01,BUSSY-SAINT-GEORGES,1,3,0,JOHV,1233
6,2015-01-01,CERGY LE HAUT,1,3,0,JOHV,896
7,2015-01-01,CERGY-PREFECTURE,1,3,0,JOHV,1307
8,2015-01-01,CERGY-SAINT-CHRISTOPHE,1,3,0,JOHV,906
9,2015-01-01,CHAMPIGNY,1,3,0,JOHV,1571


## PROFIL

In [30]:
df_profil.head()

,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_ZDC,CAT_JOUR,TRNC_HORR_60,pourc_validations,CODE_STIF_RES_NUM,PERIODE,Pourcentage_validations,ID_REFA_LDA,ï»¿CODE_STIF_TRNS,lda
0,810.0,801,116,BRY-SUR-MARNE,73166.0,DIJFP,0H-1H,"0,35",801.0,2024_S1,NaN,NaN,NaN,NaN
1,810.0,801,116,BRY-SUR-MARNE,73166.0,DIJFP,10H-11H,"7,21",801.0,2024_S1,NaN,NaN,NaN,NaN
2,810.0,801,116,BRY-SUR-MARNE,73166.0,DIJFP,11H-12H,"7,48",801.0,2024_S1,NaN,NaN,NaN,NaN
3,810.0,801,116,BRY-SUR-MARNE,73166.0,DIJFP,12H-13H,"7,4",801.0,2024_S1,NaN,NaN,NaN,NaN
4,810.0,801,116,BRY-SUR-MARNE,73166.0,DIJFP,13H-14H,"8,16",801.0,2024_S1,NaN,NaN,NaN,NaN


In [31]:
df_profil.isnull().sum()

,0
CODE_STIF_TRNS,5401
CODE_STIF_RES,0
CODE_STIF_ARRET,0
LIBELLE_ARRET,0
ID_ZDC,82274
CAT_JOUR,0
TRNC_HORR_60,0
pourc_validations,10616
CODE_STIF_RES_NUM,0
PERIODE,0


In [32]:
# Colonnes à garder pour les pics horaires (PROFIL)
cols_profil = ['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'TRNC_HORR_60', 'pourc_validations']
df_profil = df_profil[cols_profil].copy()

print("Datasets allégés ! Colonnes restantes :", df_profil.columns.tolist())

Datasets allégés ! Colonnes restantes : ['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'TRNC_HORR_60', 'pourc_validations']


In [33]:
df_profil.head()

,PERIODE,LIBELLE_ARRET,CAT_JOUR,TRNC_HORR_60,pourc_validations
0,2024_S1,BRY-SUR-MARNE,DIJFP,0H-1H,"0,35"
1,2024_S1,BRY-SUR-MARNE,DIJFP,10H-11H,"7,21"
2,2024_S1,BRY-SUR-MARNE,DIJFP,11H-12H,"7,48"
3,2024_S1,BRY-SUR-MARNE,DIJFP,12H-13H,"7,4"
4,2024_S1,BRY-SUR-MARNE,DIJFP,13H-14H,"8,16"


In [34]:
df_profil.isnull().sum()

,0
PERIODE,0
LIBELLE_ARRET,0
CAT_JOUR,0
TRNC_HORR_60,0
pourc_validations,10616


In [35]:
# Sélection des colonnes stratégiques incluant la période
colonnes_utiles = ['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'TRNC_HORR_60', 'pourc_validations']

# On crée la version allégée
df_profil = df_profil[colonnes_utiles].copy()

# Nettoyage des textes
df_profil['LIBELLE_ARRET'] = df_profil['LIBELLE_ARRET'].str.strip()
df_profil['CAT_JOUR'] = df_profil['CAT_JOUR'].str.strip()

print("Colonnes conservées :", df_profil.columns.tolist())

Colonnes conservées : ['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'TRNC_HORR_60', 'pourc_validations']


In [36]:
df_profil.head()

,PERIODE,LIBELLE_ARRET,CAT_JOUR,TRNC_HORR_60,pourc_validations
0,2024_S1,BRY-SUR-MARNE,DIJFP,0H-1H,"0,35"
1,2024_S1,BRY-SUR-MARNE,DIJFP,10H-11H,"7,21"
2,2024_S1,BRY-SUR-MARNE,DIJFP,11H-12H,"7,48"
3,2024_S1,BRY-SUR-MARNE,DIJFP,12H-13H,"7,4"
4,2024_S1,BRY-SUR-MARNE,DIJFP,13H-14H,"8,16"


In [37]:
df_profil.isnull().sum()

,0
PERIODE,0
LIBELLE_ARRET,0
CAT_JOUR,0
TRNC_HORR_60,0
pourc_validations,10616


In [38]:
# Tri du fichier des profils par Période, puis Gare, puis Heure
df_profil = df_profil.sort_values(
    by=['PERIODE', 'LIBELLE_ARRET', 'TRNC_HORR_60'],
    ascending=[True, True, True]
).reset_index(drop=True)

print("Tri du dataset PROFIL terminé !")
df_profil.head()

Tri du dataset PROFIL terminé !


,PERIODE,LIBELLE_ARRET,CAT_JOUR,TRNC_HORR_60,pourc_validations
0,2015S1,ACHERES-GRAND-CORMIER,DIJFP,0H-1H,0.12
1,2015S1,ACHERES-GRAND-CORMIER,JOHV,0H-1H,0.09
2,2015S1,ACHERES-GRAND-CORMIER,JOVS,0H-1H,0.28
3,2015S1,ACHERES-GRAND-CORMIER,SAHV,0H-1H,0.63
4,2015S1,ACHERES-GRAND-CORMIER,DIJFP,10H-11H,2.06


In [39]:
# Définition du chemin de sauvegarde
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

df_nb_final = df_nb.copy()
df_profil_final = df_profil.copy()

# Sauvegarde
df_nb_final.to_csv(os.path.join(output_base_path, 'RER_A_NB_Final.csv'), index=False)
df_profil_final.to_csv(os.path.join(output_base_path, 'RER_A_PROFIL_Mid_Save.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


In [40]:
print("=== 🧹 NETTOYAGE ET TRI CHRONOLOGIQUE DES PROFILS ===")

# 1. Sécurisation : On s'assure que les pourcentages sont bien des nombres pour pouvoir les additionner
df_profil_final['pourc_validations'] = df_profil_final['pourc_validations'].astype(str).str.replace(',', '.').astype(float)

# 2. AGRÉGATION (Fusion des doublons)
# On regroupe par Période, Gare, Jour et Heure, et on ADDITIONNE les pourcentages
taille_avant = len(df_profil_final)
df_profil_final = df_profil_final.groupby(
    ['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'TRNC_HORR_60'],
    as_index=False
)['pourc_validations'].sum()

print(f"🔄 Lignes fusionnées (Doublons additionnés) : {taille_avant} -> {len(df_profil_final)}")

# 3. CRÉATION D'UNE CLÉ DE TRI CHRONOLOGIQUE
# On extrait le premier chiffre avant le "H" (ex: "5H-6H" devient le chiffre 5)
def extraire_heure(tranche):
    try:
        return int(tranche.split('H')[0])
    except:
        return 99 # Valeur par défaut en cas d'erreur de format

# On crée une colonne temporaire invisible pour l'utilisateur
df_profil_final['HEURE_DEBUT_TRI'] = df_profil_final['TRNC_HORR_60'].apply(extraire_heure)

# 4. TRI MULTI-FACTEURS
# On trie dans l'ordre : Période -> Gare -> Type de Jour -> L'heure mathématique
df_profil_final = df_profil_final.sort_values(
    by=['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'HEURE_DEBUT_TRI'],
    ascending=[True, True, True, True]
).reset_index(drop=True)

# On supprime la colonne temporaire qui ne nous sert plus à rien
df_profil_final = df_profil_final.drop(columns=['HEURE_DEBUT_TRI'])

print("✅ Tri chronologique parfait terminé ! Voici un aperçu :")
df_profil_final.head(10)

=== 🧹 NETTOYAGE ET TRI CHRONOLOGIQUE DES PROFILS ===
🔄 Lignes fusionnées (Doublons additionnés) : 103614 -> 103614
✅ Tri chronologique parfait terminé ! Voici un aperçu :


,PERIODE,LIBELLE_ARRET,CAT_JOUR,TRNC_HORR_60,pourc_validations
0,2015S1,ACHERES-GRAND-CORMIER,DIJFP,0H-1H,0.12
1,2015S1,ACHERES-GRAND-CORMIER,DIJFP,1H-2H,0.12
2,2015S1,ACHERES-GRAND-CORMIER,DIJFP,4H-5H,0.85
3,2015S1,ACHERES-GRAND-CORMIER,DIJFP,5H-6H,1.70
4,2015S1,ACHERES-GRAND-CORMIER,DIJFP,6H-7H,2.31
5,2015S1,ACHERES-GRAND-CORMIER,DIJFP,7H-8H,3.76
6,2015S1,ACHERES-GRAND-CORMIER,DIJFP,8H-9H,3.16
7,2015S1,ACHERES-GRAND-CORMIER,DIJFP,9H-10H,1.94
8,2015S1,ACHERES-GRAND-CORMIER,DIJFP,10H-11H,2.06
9,2015S1,ACHERES-GRAND-CORMIER,DIJFP,11H-12H,2.43


In [41]:
# Définition du chemin de sauvegarde
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde
df_profil_final.to_csv(os.path.join(output_base_path, 'RER_A_PROFIL_Final.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


# MERGE PROFIL - NB

## Fusion

In [42]:
print("=== 🔗 FUSION TEMPORELLE ET GÉNÉRATION DES 24H ===")

# --- ÉTAPE 1 : CRÉATION DE LA CLÉ "PERIODE" DANS NB ---
def attribuer_periode(date):
    y = date.year
    m = date.month
    if y in [2015, 2016]: return f"{y}S1" if m <= 6 else f"{y}S2"
    if y == 2017 and m <= 6: return "2017S1"
    if y == 2024:
        if m <= 6: return "2024_S1"
        elif m <= 9: return "2024_T3"
        else: return "2024_T4"
    return f"{y}_S1" if m <= 6 else f"{y}_S2"

df_nb_final['PERIODE'] = df_nb_final['JOUR'].apply(attribuer_periode)


# --- ÉTAPE 2 : AGRÉGATION DE NB ET RENOMMAGE DE LA COLONNE ---
df_nb_grouped = df_nb_final.groupby(['PERIODE', 'JOUR', 'LIBELLE_ARRET', 'CAT_JOUR'])['NB_VALD'].sum().reset_index()
df_nb_grouped.rename(columns={'NB_VALD': 'NB_VALD_JOURNALIERE'}, inplace=True)
print(f"Lignes NB prêtes pour la fusion : {len(df_nb_grouped)}")


# --- ÉTAPE 2.5 (NOUVEAU) : COMPLÉTION DES 24 HEURES DANS LE PROFIL ---
# 1. On crée la liste des 24 tranches horaires exactes
tranches_24h = [f"{i}H-{i+1}H" for i in range(23)] + ["23H-0H"]
df_tranches = pd.DataFrame({'TRNC_HORR_60': tranches_24h, 'cle_temporaire': 1})

# 2. On récupère les combinaisons uniques existantes (Période + Gare + Type de jour)
profil_bases = df_profil_final[['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR']].drop_duplicates()
profil_bases['cle_temporaire'] = 1

# 3. Produit cartésien : on force l'existence des 24h pour chaque gare
profil_complet = pd.merge(profil_bases, df_tranches, on='cle_temporaire').drop('cle_temporaire', axis=1)

# 4. On ramène les vrais pourcentages existants, et on met 0 pour les heures "vides" (ex: 2h-3h du matin)
profil_complet = pd.merge(profil_complet, df_profil_final, on=['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR', 'TRNC_HORR_60'], how='left')
profil_complet['pourc_validations'] = profil_complet['pourc_validations'].fillna(0.0)


# --- ÉTAPE 3 : LA FUSION EXACTE (Avec le profil complété) ---
df_final_complet = pd.merge(
    df_nb_grouped,
    profil_complet,
    on=['PERIODE', 'LIBELLE_ARRET', 'CAT_JOUR'],
    how='inner'
)


# --- ÉTAPE 4 : CALCUL DES VALIDATIONS HORAIRES & SÉCURITÉ MINIMUM 1 ---
df_final_complet['NB_VALD_HORAIRE'] = (df_final_complet['NB_VALD_JOURNALIERE'] * df_final_complet['pourc_validations']) / 100
df_final_complet['NB_VALD_HORAIRE'] = df_final_complet['NB_VALD_HORAIRE'].round(0).astype(int)

# RÈGLE MÉTIER : Si pourcentage > 0 mais résultat tombé à 0, on force à 1
# Attention : on ne touche PAS à ceux où le pourcentage est vraiment de 0 !
condition_zero_parasite = (df_final_complet['pourc_validations'] > 0) & (df_final_complet['NB_VALD_HORAIRE'] == 0)
df_final_complet.loc[condition_zero_parasite, 'NB_VALD_HORAIRE'] = 1


# --- ÉTAPE 5 : RECALCUL ET MISE À JOUR DE LA COLONNE JOURNALIÈRE ---
df_final_complet['NB_VALD_JOURNALIERE'] = df_final_complet.groupby(
    ['PERIODE', 'JOUR', 'LIBELLE_ARRET', 'CAT_JOUR']
)['NB_VALD_HORAIRE'].transform('sum')

print(f"✅ Fusion avec 24h garanties ! Dataset final : {len(df_final_complet)} lignes.")

# On trie pour vérifier que l'ordre des heures est bon
df_final_complet.head()

=== 🔗 FUSION TEMPORELLE ET GÉNÉRATION DES 24H ===
Lignes NB prêtes pour la fusion : 59203
✅ Fusion avec 24h garanties ! Dataset final : 1219608 lignes.


,PERIODE,JOUR,LIBELLE_ARRET,CAT_JOUR,NB_VALD_JOURNALIERE,TRNC_HORR_60,pourc_validations,NB_VALD_HORAIRE
0,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,0H-1H,0.09,1
1,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,1H-2H,0.00,0
2,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,2H-3H,0.00,0
3,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,3H-4H,0.00,0
4,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,4H-5H,0.06,1


In [43]:
df_final_complet.head()

,PERIODE,JOUR,LIBELLE_ARRET,CAT_JOUR,NB_VALD_JOURNALIERE,TRNC_HORR_60,pourc_validations,NB_VALD_HORAIRE
0,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,0H-1H,0.09,1
1,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,1H-2H,0.00,0
2,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,2H-3H,0.00,0
3,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,3H-4H,0.00,0
4,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,4H-5H,0.06,1


In [44]:
import pandas as pd

print("=== 🧹 FORÇAGE DU FORMAT NUMÉRIQUE ===")

# 1. On s'assure de remplacer les éventuelles virgules françaises par des points (format US/Python)
# L'utilisation de .astype(str) permet d'éviter une erreur si certains nombres sont déjà au bon format
df_final_complet['pourc_validations'] = df_final_complet['pourc_validations'].astype(str).str.replace(',', '.')

# 2. On force la conversion en float
# L'option errors='coerce' est la sécurité du Data Engineer : si un texte bizarre (ex: "Erreur")
# s'est glissé là, il sera transformé en case vide (NaN) au lieu de faire planter tout ton script.
df_final_complet['pourc_validations'] = pd.to_numeric(df_final_complet['pourc_validations'], errors='coerce')

# 3. Vérification technique
print("Nouveau type de la colonne :", df_final_complet['pourc_validations'].dtype)

# On affiche les lignes où la conversion aurait pu échouer (si le résultat affiche 0, ton fichier est parfait !)
erreurs = df_final_complet['pourc_validations'].isna().sum()
print(f"Nombre de valeurs non reconnues après conversion : {erreurs}")

=== 🧹 FORÇAGE DU FORMAT NUMÉRIQUE ===
Nouveau type de la colonne : float64
Nombre de valeurs non reconnues après conversion : 0


In [45]:
# --- EXPORTATION DU DATASET AU FORMAT STANDARD INTERNATIONAL ---
chemin_export = 'RER_A_DATAMART_FINAL_US.csv' # Ajuste ton chemin Google Drive si nécessaire

df_final_complet.to_csv(
    chemin_export,
    index=False,
    sep=',',
    decimal='.'
)

print("🎯 Pipeline terminé ! Le fichier CSV est purement international (séparateur ',' et décimal '.').")
print("Toutes les valeurs de la colonne pourc_validations possèdent strictement un point.")

🎯 Pipeline terminé ! Le fichier CSV est purement international (séparateur ',' et décimal '.').
Toutes les valeurs de la colonne pourc_validations possèdent strictement un point.


In [46]:
#Correction des intitulés des gares (car les noms ont changé avec le temps, au moins dans la db)
dictionnaire_correction_gares = {
    "ACHERES GRAND": "ACHERES-GRAND-CORMIER",
    "ACHERES VILLE": "ACHERES-VILLE",
    "BOISSY-ST-LEG.": "BOISSY-SAINT-LEGER",
    "BUSSY-ST-GEOR.": "BUSSY-SAINT-GEORGES",
    "CERGY PREFECTUR": "CERGY-PREFECTURE",
    "CERGY ST CHRIST": "CERGY-SAINT-CHRISTOPHE",
    "CH.D.G.ETOILE": "CHARLES DE GAULLE ETOILE",
    "CHATELET": "CHATELET-LES HALLES",
    "CONFLANS FO": "CONFLANS-FIN-D'OISE",
    "FONTENAY-S-B.": "FONTENAY-SOUS-BOIS",
    "HOUILLES": "HOUILLES-CARRIERES-SUR-SEINE",
    "JOINVILLE-LE-P": "JOINVILLE-LE-PONT",
    "LA DEFENSE": "LA DEFENSE-GRANDE ARCHE",
    "LA VARENNE-CH.": "LA VARENNE-CHENNEVIERES",
    "LE PARC S.MAUR": "LE PARC-DE-SAINT-MAUR",
    "LE VESINET-CEN": "LE VESINET-CENTRE",
    "LE VESINET-L.P": "LE VESINET-LE PECQ",
    "M.L.V.CHESSY": "CHESSY - MARNE-LA-VALLEE",
    "MAISONS LAFFITT": "MAISONS-LAFFITTE",
    "NANTERRE-PREF.": "NANTERRE-PREFECTURE",
    "NANTERRE-UNIV.": "NANTERRE-UNIVERSITE",
    "NEUILLY-PLAIS.": "NEUILLY-PLAISANCE"
}

# Application du nettoyage sur le Dataset
df_final_complet['LIBELLE_ARRET'] = df_final_complet['LIBELLE_ARRET'].replace(dictionnaire_correction_gares)

## Persistance de la fusion RER A

In [47]:
# Définir un chemin de base pour la sauvegarde dans le nouveau dossier
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde des versions finales "propres"
df_final_complet.to_csv(os.path.join(output_base_path, 'RER_A_Final.csv'), index=False)

print("Fichiers sauvegardés dans le dossier YBOOST DATA B2 ! Tu peux maintenant les charger comme un CSV classique.")

Fichiers sauvegardés dans le dossier YBOOST DATA B2 ! Tu peux maintenant les charger comme un CSV classique.


In [48]:
print(len(df_nb_final))
print(len(df_profil_final))
print(len(df_final_complet))

59203
103614
1219608


# Météo

## Création de la liste des sations du RER A

In [49]:
print("=== 🏙️ EXTRACTION DES STATIONS / VILLES DU RER A ===")

# --- OPTION 1 : Depuis le DataFrame en mémoire ---
# Si df_final_complet existe encore dans ta session :
if 'df_final_complet' in locals():
    df_source = df_final_complet
else:
    # --- OPTION 2 : Depuis le fichier CSV sauvegardé ---
    chemin_fichier = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Final.csv'
    print(f"Chargement du fichier : {chemin_fichier}")
    df_source = pd.read_csv(chemin_fichier, sep=',')

# 1. Extraction des valeurs uniques de la colonne des arrêts
# .unique() élimine les doublons, .tolist() convertit le résultat en liste Python standard
liste_villes_gares = df_source['LIBELLE_ARRET'].unique().tolist()

# 2. Tri par ordre alphabétique pour faciliter la lecture et la recherche
liste_villes_gares.sort()

# 3. Affichage des résultats
print(f"\n📊 Nombre total de gares/villes uniques : {len(liste_villes_gares)}")
print("-" * 50)
print("Liste des gares disponibles pour la correspondance météo :")
for station in liste_villes_gares:
    print(f"📍 {station}")
print("-" * 50)

=== 🏙️ EXTRACTION DES STATIONS / VILLES DU RER A ===

📊 Nombre total de gares/villes uniques : 54
--------------------------------------------------
Liste des gares disponibles pour la correspondance météo :
📍 ACHERES-GRAND-CORMIER
📍 ACHERES-VILLE
📍 AUBER
📍 BOISSY-SAINT-LEGER
📍 BRY-SUR-MARNE
📍 BUSSY-SAINT-GEORGES
📍 CERGY LE HAUT
📍 CERGY-PREFECTURE
📍 CERGY-SAINT-CHRISTOPHE
📍 CHAMPIGNY
📍 CHARLES DE GAULLE ETOILE
📍 CHATELET-LES HALLES
📍 CHATOU-CROISSY
📍 CHESSY - MARNE-LA-VALLEE
📍 CONFLANS-FIN-D'OISE
📍 FONTENAY-SOUS-BOIS
📍 GARE DE LYON
📍 HOUILLES-CARRIERES-SUR-SEINE
📍 JOINVILLE-LE-PONT
📍 LA DEFENSE-GRANDE ARCHE
📍 LA VARENNE-CHENNEVIERES
📍 LE PARC-DE-SAINT-MAUR
📍 LE VESINET-CENTRE
📍 LE VESINET-LE PECQ
📍 LOGNES
📍 MAISONS-LAFFITTE
📍 NANTERRE-PREFECTURE
📍 NANTERRE-UNIVERSITE
📍 NANTERRE-VILLE
📍 NATION
📍 NEUILLY-PLAISANCE
📍 NEUVILLE UNIVER
📍 NEUVILLE UNIVERSITE
📍 NOGENT-S-MARNE
📍 NOGENT-SUR-MARNE
📍 NOISIEL
📍 NOISY-CHAMPS
📍 NOISY-LE-GRAND
📍 NOISY-LE-GRAND-MONT D'EST
📍 POISSY
📍 RUEIL-MALMAIS.
📍 RU

## Météo - Import des datasets d'analyse météorologique depuis 2015 jusqu'à 2024

In [50]:
all_meteo_data = []
total_files = 0
correct_files = 0
failed_files = 0

path_racine = '/content/drive/MyDrive/YBOOST DATA B2/csv/'
path_meteo = os.path.join(path_racine, 'Météo/Météo/')

# --- MAPPING INTELLIGENT ---
# On classe tes villes par département, pour savoir par quelle ville remplacé si on a pas de données pour la ville d'un arrêt.
villes_par_dept = {
    "75": ["PARIS"],
    "77": ["BUSSY-SAINT-GEORGES", "TORCY"],
    "78": ["ACHERES", "CHATOU", "CONFLANS", "HOUILLES", "MAISONS-LAFFITTE", "POISSY", "SAINT-GERMAIN-EN-LAYE", "SARTROUVILLE"],
    "92": ["NANTERRE"],
    "93": ["NEUILLY-PLAISANCE"],
    "94": ["BRY-SUR-MARNE", "CHAMPIGNY", "FONTENAY-SOUS-BOIS", "JOINVILLE", "VINCENNES"],
    "95": ["CERGY"]
}

#De 2015 à 2025 (dans le final, 2025 ne sera pas pris car les données rer A sont jusqu'à 2024 mais au cas où)
annees_cibles = [str(annee) for annee in range(2015, 2025)]

print("=== 🌤️ DÉMARRAGE DU SCANNER DE DONNÉES MÉTÉO ===")

for root, dirs, files in os.walk(path_meteo):
    print(f"\n📂 --- Entrée dans le dossier : {os.path.basename(root)} ---")

    for file in files:
        if "RR-T-Vent" in file and (file.endswith('.csv') or file.endswith('.csv.gz')):
            total_files += 1
            file_path = os.path.join(root, file)

            # 1. On détecte le département grâce au nom du fichier
            dept_fichier = None
            for dept in villes_par_dept.keys():
                if f"_{dept}_" in file:
                    dept_fichier = dept
                    break

            print(f"  🔍 Vérification de : {file} (Département: {dept_fichier})")

            if not dept_fichier:
                print(f"     ⚠️ Ignoré : Ce département n'est pas sur le RER A.")
                continue

            # 2. Chargement du fichier
            df_full = None
            try:
                df_full = pd.read_csv(file_path, sep=';', engine='python', encoding='utf-8', dtype=str, on_bad_lines='skip')
                df_full.columns = [c.strip().upper() for c in df_full.columns]
            except UnicodeDecodeError:
                try:
                    df_full = pd.read_csv(file_path, sep=';', engine='python', encoding='latin1', dtype=str, on_bad_lines='skip')
                    df_full.columns = [c.strip().upper() for c in df_full.columns]
                except Exception as e:
                    failed_files += 1
                    continue
            except Exception as e:
                failed_files += 1
                continue

            if df_full is not None and 'NOM_USUEL' in df_full.columns:

                # 3. Filtre Temporel (2015-2024)
                colonne_date = 'AAAAMMJJ' if 'AAAAMMJJ' in df_full.columns else 'AAAAMM'
                if colonne_date in df_full.columns:
                    df_full['ANNEE_EXTRAITE'] = df_full[colonne_date].str[:4]
                    df_recent = df_full[df_full['ANNEE_EXTRAITE'].isin(annees_cibles)].copy()
                    df_recent = df_recent.drop(columns=['ANNEE_EXTRAITE'])
                else:
                    failed_files += 1
                    continue

                if df_recent.empty:
                    print(f"     ⚠️ VIDE : Aucune donnée pour 2015-2026.")
                    failed_files += 1
                    continue

                # --- 4. LOGIQUE D'ATTRIBUTION (VILLE_CIBLE) ---
                villes_a_chercher = villes_par_dept[dept_fichier]
                stations_disponibles = df_recent['NOM_USUEL'].dropna().unique()

                if len(stations_disponibles) == 0:
                    failed_files += 1
                    continue

                station_secours = stations_disponibles[0] # La station fallback du département
                lignes_sauvegardees = 0

                # On boucle sur chaque ville cible de ce département spécifique
                for ville in villes_a_chercher:
                    masque_ville = df_recent['NOM_USUEL'].str.contains(ville, case=False, na=False)
                    df_ville = df_recent[masque_ville].copy()

                    if not df_ville.empty:
                        # Cas A : La ville exacte est trouvée
                        df_ville['VILLE_CIBLE'] = ville
                        all_meteo_data.append(df_ville)
                        lignes_sauvegardees += len(df_ville)
                        print(f"     ✅🎯 [EXACT] Station {df_ville['NOM_USUEL'].iloc[0]} -> rattachée à {ville}")
                    else:
                        # Cas B : On prend la station de secours ET on indique la ville qu'elle remplace
                        df_fallback = df_recent[df_recent['NOM_USUEL'] == station_secours].copy()
                        df_fallback['VILLE_CIBLE'] = ville
                        all_meteo_data.append(df_fallback)
                        lignes_sauvegardees += len(df_fallback)
                        print(f"     🔄 [SECOURS] Station {station_secours} -> rattachée à {ville}")

                if lignes_sauvegardees > 0:
                    correct_files += 1
                else:
                    failed_files += 1

            else:
                failed_files += 1

if all_meteo_data:
    df_meteo_final = pd.concat(all_meteo_data, ignore_index=True)
    print(f"\n" + "="*50)
    print(f"=== 📊 RÉSUMÉ FINAL DU TRAITEMENT MÉTÉO ===")
    print(f"📁 Total de fichiers examinés : {total_files}")
    print(f"✅ Fichiers traités avec succès : {correct_files}")
    print(f"📈 Total de lignes MÉTÉO fusionnées : {len(df_meteo_final)}")
    print("="*50)
else:
    print("\n❌ ERREUR : Aucune donnée météo n'a pu être chargée.")

=== 🌤️ DÉMARRAGE DU SCANNER DE DONNÉES MÉTÉO ===

📂 --- Entrée dans le dossier :  ---
  🔍 Vérification de : Q_75_previous-1950-2024_RR-T-Vent.csv (Département: 75)
     ✅🎯 [EXACT] Station PARIS-MONTSOURIS -> rattachée à PARIS
  🔍 Vérification de : Q_77_previous-1950-2024_RR-T-Vent.csv (Département: 77)
     🔄 [SECOURS] Station ARBONNE -> rattachée à BUSSY-SAINT-GEORGES
     ✅🎯 [EXACT] Station TORCY -> rattachée à TORCY
  🔍 Vérification de : Q_78_previous-1950-2024_RR-T-Vent.csv (Département: 78)
     ✅🎯 [EXACT] Station ACHERES -> rattachée à ACHERES
     🔄 [SECOURS] Station ABLIS -> rattachée à CHATOU
     🔄 [SECOURS] Station ABLIS -> rattachée à CONFLANS
     🔄 [SECOURS] Station ABLIS -> rattachée à HOUILLES
     🔄 [SECOURS] Station ABLIS -> rattachée à MAISONS-LAFFITTE
     🔄 [SECOURS] Station ABLIS -> rattachée à POISSY
     🔄 [SECOURS] Station ABLIS -> rattachée à SAINT-GERMAIN-EN-LAYE
     🔄 [SECOURS] Station ABLIS -> rattachée à SARTROUVILLE
  🔍 Vérification de : Q_92_previous-19

In [51]:
# Définir un chemin de base pour la sauvegarde dans le nouveau dossier
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde des versions finales "propres"
df_meteo_final.to_csv(os.path.join(output_base_path, 'RER_A_Meteo_Base.csv'), index=False)

print("Fichiers sauvegardés dans le dossier YBOOST DATA B2 ! Tu peux maintenant les charger comme un CSV classique.")

Fichiers sauvegardés dans le dossier YBOOST DATA B2 ! Tu peux maintenant les charger comme un CSV classique.


In [52]:
print("=== 🧹 NETTOYAGE ET SÉLECTION DES 7 COLONNES MAGIQUES ===")

chemin_meteo_final = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Meteo_Base.csv'

# 1. LECTURE BLINDÉE
try:
    # On essaie d'abord avec la virgule (format CSV standard)
    df_meteo = pd.read_csv(chemin_meteo_final, sep=',', engine='python', on_bad_lines='skip')

    # Nettoyage IMMÉDIAT des en-têtes (retire les espaces invisibles et met en majuscules)
    df_meteo.columns = df_meteo.columns.str.strip().str.upper()

    # Si ça n'a pas marché (tout est resté dans 1 seule colonne), on passe au point-virgule
    if 'NOM_USUEL' not in df_meteo.columns:
        df_meteo = pd.read_csv(chemin_meteo_final, sep=';', engine='python', on_bad_lines='skip')
        df_meteo.columns = df_meteo.columns.str.strip().str.upper()

except Exception as e:
    print(f"❌ Erreur lors de la lecture du fichier : {e}")

# 2. VÉRIFICATION DE SÉCURITÉ
print("Colonnes détectées dans le fichier :", df_meteo.columns.tolist()[:10], "...")

colonnes_utiles = ['AAAAMMJJ', 'NOM_USUEL', 'VILLE_CIBLE', 'RR', 'TN', 'TX', 'FXI']
colonnes_manquantes = [col for col in colonnes_utiles if col not in df_meteo.columns]

if colonnes_manquantes:
    print(f"❌ ERREUR CRITIQUE : Il manque ces colonnes dans le fichier : {colonnes_manquantes}")
else:
    # 3. FILTRAGE ET CONVERSION
    df_clean = df_meteo[colonnes_utiles].copy()

    # On force la date en texte brut avant de la convertir pour éviter les bugs
    df_clean['AAAAMMJJ'] = pd.to_datetime(df_clean['AAAAMMJJ'].astype(str), format='%Y%m%d', errors='coerce')

    for col in ['RR', 'TN', 'TX', 'FXI']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    # 4. CRÉATION DES COLONNES SUR-MESURE POUR TON IA
    df_clean['TEMP_MOY_C'] = (df_clean['TN'] + df_clean['TX']) / 2

    def eval_meteo_globale(row):
        pluie = row['RR']
        vent = row['FXI']
        temp = row['TEMP_MOY_C']

        if pd.isna(pluie) and pd.isna(temp):
            return 'Inconnu'

        # Conditions extrêmes
        if (pd.notna(vent) and vent > 22.0) or (pd.notna(pluie) and pluie > 15.0):
            return 'Conditions Extrêmes'

        # Températures
        if pd.notna(temp):
            if temp <= 0:
                return 'Gel/Froid Extrême'
            if temp >= 30:
                return 'Canicule'

        # Précipitations
        if pd.notna(pluie):
            if pluie > 2.0:
                return 'Pluvieux'
            elif 0 < pluie <= 2.0:
                return 'Pluie Légère'

        # Vent
        if pd.notna(vent) and vent > 15.0:
            return 'Venteux'

        return 'Temps Calme'

    df_clean['METEO_GLOBALE'] = df_clean.apply(eval_meteo_globale, axis=1)

    # 5. ASSEMBLAGE ET RENOMMAGE
    df_meteo_final = df_clean[[
        'AAAAMMJJ', 'NOM_USUEL', 'VILLE_CIBLE', 'METEO_GLOBALE', 'FXI', 'RR', 'TEMP_MOY_C'
    ]]

    df_meteo_final.columns = [
        'JOUR', 'STATION_METEO', 'VILLE_CIBLE', 'METEO_GLOBALE', 'VENT_RAFALE_MS', 'PLUIE_MM', 'TEMP_MOYENNE_C'
    ]

    # 6. SAUVEGARDE DU DATASET PROPRE
    chemin_propre = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Meteo_Nettoyee.csv'
    df_meteo_final.to_csv(chemin_propre, index=False, sep=';', decimal=',')

    print(f"\n✅ Nettoyage terminé ! Les données sont enregistrées.")
    print("🔍 Aperçu du résultat :")
    print(df_meteo_final.head())

=== 🧹 NETTOYAGE ET SÉLECTION DES 7 COLONNES MAGIQUES ===
Colonnes détectées dans le fichier : ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR', 'TN', 'QTN'] ...

✅ Nettoyage terminé ! Les données sont enregistrées.
🔍 Aperçu du résultat :
        JOUR     STATION_METEO VILLE_CIBLE METEO_GLOBALE  VENT_RAFALE_MS  \
0 2015-01-01  PARIS-MONTSOURIS       PARIS   Temps Calme            10.3   
1 2015-01-02  PARIS-MONTSOURIS       PARIS   Temps Calme             9.9   
2 2015-01-03  PARIS-MONTSOURIS       PARIS      Pluvieux            18.0   
3 2015-01-04  PARIS-MONTSOURIS       PARIS   Temps Calme             9.9   
4 2015-01-05  PARIS-MONTSOURIS       PARIS   Temps Calme             5.0   

   PLUIE_MM  TEMP_MOYENNE_C  
0       0.0            2.95  
1       0.0            6.75  
2       4.8            7.00  
3       0.0            5.30  
4       0.0            1.15  


In [53]:
print("=== 🩹 COMBLEMENT DES DONNÉES MANQUANTES ===")

chemin_nettoye = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Meteo_Nettoyee.csv'

try:
    df_meteo_final = pd.read_csv(chemin_nettoye, sep=';', decimal=',')
except Exception as e:
    df_meteo_final = pd.read_csv(chemin_nettoye, sep=',')

# Renommage de sécurité
df_meteo_final = df_meteo_final.rename(columns={'FXI': 'VENT_RAFALE_MS', 'RR': 'PLUIE_MM', 'TEMP_MOY_C': 'TEMP_MOYENNE_C'})
df_meteo_final['JOUR'] = pd.to_datetime(df_meteo_final['JOUR'], errors='coerce')

print(f"⚠️ Nombre de cases vides avant traitement : {df_meteo_final.isnull().sum().sum()}")

# 1. Tri par ville et date
df_meteo_final = df_meteo_final.sort_values(by=['VILLE_CIBLE', 'JOUR'])
colonnes_num = ['VENT_RAFALE_MS', 'PLUIE_MM', 'TEMP_MOYENNE_C']

# --- ÉTAPE A : Imputation Locale (On bouche les petits trous avec la veille) ---
df_meteo_final[colonnes_num + ['METEO_GLOBALE']] = df_meteo_final.groupby('VILLE_CIBLE')[colonnes_num + ['METEO_GLOBALE']].ffill().bfill()

# --- ÉTAPE B : Imputation Régionale (La solution pour tes 19 874 trous !) ---
# Si une ville n'a aucun capteur pour une donnée, on prend la moyenne de l'Île-de-France CE JOUR-LÀ.
for col in colonnes_num:
    moyenne_du_jour = df_meteo_final.groupby('JOUR')[col].transform('mean')
    df_meteo_final[col] = df_meteo_final[col].fillna(moyenne_du_jour)

# --- ÉTAPE C : Imputation Globale (Sécurité ultime) ---
# S'il y a eu un jour entier où AUCUNE station n'a fonctionné
df_meteo_final = df_meteo_final.sort_values(by='JOUR')
df_meteo_final[colonnes_num] = df_meteo_final[colonnes_num].ffill().bfill()

# --- ÉTAPE D : Recalcul de la Météo Globale ---
# Maintenant qu'on a réparé les chiffres de vent/pluie, on met à jour le texte
def reparer_meteo(row):
    if pd.notna(row['METEO_GLOBALE']) and row['METEO_GLOBALE'] != 'Inconnu':
        return row['METEO_GLOBALE']

    pluie = row['PLUIE_MM']
    vent = row['VENT_RAFALE_MS']
    temp = row['TEMP_MOYENNE_C']

    if vent > 22.0 or pluie > 15.0: return 'Conditions Extrêmes'
    if temp <= 0: return 'Gel/Froid Extrême'
    if temp >= 30: return 'Canicule'
    if pluie > 2.0: return 'Pluvieux'
    if 0 < pluie <= 2.0: return 'Pluie Légère'
    if vent > 15.0: return 'Venteux'
    return 'Temps Calme'

df_meteo_final['METEO_GLOBALE'] = df_meteo_final.apply(reparer_meteo, axis=1)

print(f"✅ Nombre de cases vides après traitement : {df_meteo_final.isnull().sum().sum()}")

=== 🩹 COMBLEMENT DES DONNÉES MANQUANTES ===
⚠️ Nombre de cases vides avant traitement : 53818
✅ Nombre de cases vides après traitement : 0


## Persistance du csv Météo final

In [54]:
# Définition du chemin de sauvegarde
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde
df_meteo_final.to_csv(os.path.join(output_base_path, 'RER_A_Meteo_Final.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


c'est à jour


# MERGE FINAL ( RER & METEO )

## Verif finale des fichiers à fusionner

In [55]:
df_final_complet.isnull().sum()

,0
PERIODE,0
JOUR,0
LIBELLE_ARRET,0
CAT_JOUR,0
NB_VALD_JOURNALIERE,0
TRNC_HORR_60,0
pourc_validations,0
NB_VALD_HORAIRE,0


In [56]:
df_meteo_final.isnull().sum()

,0
JOUR,0
STATION_METEO,0
VILLE_CIBLE,0
METEO_GLOBALE,0
VENT_RAFALE_MS,0
PLUIE_MM,0
TEMP_MOYENNE_C,0


In [57]:
print(len(df_final_complet))
print(len(df_meteo_final))

1219608
47283


In [58]:
# Définition du chemin de sauvegarde
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde
df_final_complet.to_csv(os.path.join(output_base_path, 'RER_A_DATA_Final.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


In [59]:
# Définition du chemin de sauvegarde
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde
df_meteo_final.to_csv(os.path.join(output_base_path, 'RER_A_Meteo_Final.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


## Fusion

In [60]:
print("=== 🚀 LA FUSION FINALE (LE DATAMART) ===")

# 1. CHEMINS DES FICHIERS
chemin_trafic = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_DATA_Final.csv'
chemin_meteo_complete = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Meteo_Final.csv'
chemin_datamart = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Final_FINAL.csv'

# 2. CHARGEMENT ROBUSTE DES DONNÉES
print("📂 Chargement des fichiers...")
try:
    df_trafic = pd.read_csv(chemin_trafic, sep=',', on_bad_lines='skip')
    if 'LIBELLE_ARRET' not in df_trafic.columns:
        df_trafic = pd.read_csv(chemin_trafic, sep=';', on_bad_lines='skip')

    df_meteo = pd.read_csv(chemin_meteo_complete, sep=';', decimal=',', on_bad_lines='skip')
    if 'VILLE_CIBLE' not in df_meteo.columns:
        df_meteo = pd.read_csv(chemin_meteo_complete, sep=',', on_bad_lines='skip')
except Exception as e:
    print(f"❌ Erreur de lecture : {e}")

# Préparation des dates pour que Pandas puisse les superposer
df_trafic['JOUR'] = pd.to_datetime(df_trafic['JOUR'], errors='coerce')
df_meteo['JOUR'] = pd.to_datetime(df_meteo['JOUR'], errors='coerce')


# 3. LE DICTIONNAIRE DE MAPPING (Gare RER -> Ville Météo)
print("🗺️ Attribution des gares RER à leurs zones Météo...")

# --- OPTIMISATION : On sort les variables pour ne les calculer qu'une seule fois ---
villes_meteo_uniques = df_meteo['VILLE_CIBLE'].dropna().unique().tolist()
gares_paris = ['AUBER', 'CHARLES DE GAULLE ETOILE', 'CH.D.G.ETOILE', 'CHATELET-LES HALLES', 'CHATELET', 'GARE DE LYON', 'NATION']
fallbacks = {
    'LA DEFENSE-GRANDE ARCHE': 'NANTERRE', 'LA DEFENSE': 'NANTERRE',
    'BOISSY-SAINT-LEGER': 'CHAMPIGNY', 'BOISSY-ST-LEG.': 'CHAMPIGNY',
    'CHESSY - MARNE-LA-VALLEE': 'TORCY', 'M.L.V.CHESSY': 'TORCY',
    'LA VARENNE-CHENNEVIERES': 'CHAMPIGNY', 'LA VARENNE-CH.': 'CHAMPIGNY',
    'LE PARC-DE-SAINT-MAUR': 'CHAMPIGNY', 'LE PARC S.MAUR': 'CHAMPIGNY',
    'LE VESINET-CENTRE': 'CHATOU', 'LE VESINET-CEN': 'CHATOU',
    'LE VESINET-LE PECQ': 'CHATOU', 'LE VESINET-L.P': 'CHATOU',
    'LOGNES': 'TORCY',
    'NOGENT-SUR-MARNE': 'FONTENAY-SOUS-BOIS', 'NOGENT-S-MARNE': 'FONTENAY-SOUS-BOIS',
    'NOISIEL': 'TORCY', 'NOISY-CHAMPS': 'TORCY',
    "NOISY-LE-GRAND-MONT D'EST": 'NEUILLY-PLAISANCE', 'NOISY-LE-GRAND': 'NEUILLY-PLAISANCE',
    'RUEIL-MALMAISON': 'NANTERRE', 'RUEIL-MALMAIS.': 'NANTERRE',
    'SAINT-MAUR-CRETEIL': 'CHAMPIGNY', 'ST-MAUR-CRET.': 'CHAMPIGNY',
    'SUCY-BONNEUIL': 'CHAMPIGNY',
    "VAL-D'EUROPE": 'BUSSY-SAINT-GEORGES', "VAL D'EUROPE": 'BUSSY-SAINT-GEORGES',
    'VAL-DE-FONTENAY': 'FONTENAY-SOUS-BOIS', 'VAL D.FONTENAY': 'FONTENAY-SOUS-BOIS',
    'NEUVILLE UNIVERSITE': 'CERGY', 'NEUVILLE UNIVER': 'CERGY'
}

def trouver_ville_meteo(gare):
    if gare in gares_paris: return 'PARIS'
    if 'ST-GERMAIN' in gare: return 'SAINT-GERMAIN-EN-LAYE'
    for ville in villes_meteo_uniques:
        if ville in gare: return ville
    return fallbacks.get(gare, 'PARIS')

# --- OPTIMISATION : On calcule le mapping uniquement sur les gares uniques (~50 calculs) ---
gares_uniques = df_trafic['LIBELLE_ARRET'].unique()
dictionnaire_mapping_rapide = {gare: trouver_ville_meteo(gare) for gare in gares_uniques}

# On applique le dictionnaire pré-calculé à toute la colonne (instantané)
df_trafic['VILLE_CIBLE'] = df_trafic['LIBELLE_ARRET'].map(dictionnaire_mapping_rapide)


# 4. SÉCURITÉ MÉTÉO
print("🧹 Nettoyage des doublons météo...")
df_meteo = df_meteo.drop_duplicates(subset=['JOUR', 'VILLE_CIBLE'], keep='first')


# 5. LA FUSION FINALE !
print("🔗 Croisement des bases de données...")
df_merged_final = pd.merge(
    df_trafic,
    df_meteo,
    on=['JOUR', 'VILLE_CIBLE'],
    how='left'
)

# On remet le dataset dans l'ordre chronologique pour que ce soit propre
df_merged_final = df_merged_final.sort_values(by=['JOUR', 'LIBELLE_ARRET', 'TRNC_HORR_60']).reset_index(drop=True)

# Test de qualité ultime : A-t-on des lignes sans météo ?
trous_restants = df_merged_final['METEO_GLOBALE'].isnull().sum()
if trous_restants == 0:
    print("✅ QUALITÉ 100% : Aucune ligne météo manquante !")
else:
    print(f"⚠️ Attention : il reste {trous_restants} lignes de RER sans météo.")


# 6. SAUVEGARDE
df_merged_final.to_csv(chemin_datamart, index=False, sep=';', decimal=',')

print(f"\n🎉 SUCCÈS ABSOLU ! Fichier final est généré : {len(df_merged_final)} lignes parfaites.")
print("Aperçu des premières lignes :")
print(df_merged_final[['JOUR', 'LIBELLE_ARRET', 'TRNC_HORR_60', 'NB_VALD_HORAIRE', 'METEO_GLOBALE', 'TEMP_MOYENNE_C']].head())

=== 🚀 LA FUSION FINALE (LE DATAMART) ===
📂 Chargement des fichiers...
🗺️ Attribution des gares RER à leurs zones Météo...
🧹 Nettoyage des doublons météo...
🔗 Croisement des bases de données...
⚠️ Attention : il reste 358968 lignes de RER sans météo.

🎉 SUCCÈS ABSOLU ! Fichier final est généré : 1219608 lignes parfaites.
Aperçu des premières lignes :
        JOUR          LIBELLE_ARRET TRNC_HORR_60  NB_VALD_HORAIRE  \
0 2015-01-01  ACHERES-GRAND-CORMIER        0H-1H                1   
1 2015-01-01  ACHERES-GRAND-CORMIER      10H-11H                1   
2 2015-01-01  ACHERES-GRAND-CORMIER      11H-12H                1   
3 2015-01-01  ACHERES-GRAND-CORMIER      12H-13H                1   
4 2015-01-01  ACHERES-GRAND-CORMIER      13H-14H                1   

  METEO_GLOBALE  TEMP_MOYENNE_C  
0   Temps Calme             0.3  
1   Temps Calme             0.3  
2   Temps Calme             0.3  
3   Temps Calme             0.3  
4   Temps Calme             0.3  


In [61]:
df_merged_final.isnull().sum()

,0
PERIODE,0
JOUR,0
LIBELLE_ARRET,0
CAT_JOUR,0
NB_VALD_JOURNALIERE,0
TRNC_HORR_60,0
pourc_validations,0
NB_VALD_HORAIRE,0
VILLE_CIBLE,0
STATION_METEO,358968


In [62]:
print("=== 🚀 SCRIPT 4 : LA FUSION FINALE (LE DATAMART) ===")

# 1. CHEMINS DES FICHIERS
chemin_trafic = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Final.csv'
chemin_meteo_complete = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Meteo_Final.csv'
chemin_datamart = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Final_FINAL.csv'

# 2. CHARGEMENT ROBUSTE DES DONNÉES
print("📂 Chargement des fichiers...")
try:
    df_trafic = pd.read_csv(chemin_trafic, sep=',', on_bad_lines='skip')
    if 'LIBELLE_ARRET' not in df_trafic.columns:
        df_trafic = pd.read_csv(chemin_trafic, sep=';', on_bad_lines='skip')

    df_meteo = pd.read_csv(chemin_meteo_complete, sep=';', decimal=',', on_bad_lines='skip')
    if 'VILLE_CIBLE' not in df_meteo.columns:
        df_meteo = pd.read_csv(chemin_meteo_complete, sep=',', on_bad_lines='skip')
except Exception as e:
    print(f"❌ Erreur de lecture : {e}")

# Préparation des dates pour que Pandas puisse les superposer
df_trafic['JOUR'] = pd.to_datetime(df_trafic['JOUR'], errors='coerce')
df_meteo['JOUR'] = pd.to_datetime(df_meteo['JOUR'], errors='coerce')


# 3. LE DICTIONNAIRE DE MAPPING (Gare RER -> Ville Météo)
print("🗺️ Attribution des gares RER à leurs zones Météo...")

villes_meteo_uniques = df_meteo['VILLE_CIBLE'].dropna().unique().tolist()
gares_paris = ['AUBER', 'CHARLES DE GAULLE ETOILE', 'CH.D.G.ETOILE', 'CHATELET-LES HALLES', 'CHATELET', 'GARE DE LYON', 'NATION']
fallbacks = {
    'LA DEFENSE-GRANDE ARCHE': 'NANTERRE', 'LA DEFENSE': 'NANTERRE',
    'BOISSY-SAINT-LEGER': 'CHAMPIGNY', 'BOISSY-ST-LEG.': 'CHAMPIGNY',
    'CHESSY - MARNE-LA-VALLEE': 'TORCY', 'M.L.V.CHESSY': 'TORCY',
    'LA VARENNE-CHENNEVIERES': 'CHAMPIGNY', 'LA VARENNE-CH.': 'CHAMPIGNY',
    'LE PARC-DE-SAINT-MAUR': 'CHAMPIGNY', 'LE PARC S.MAUR': 'CHAMPIGNY',
    'LE VESINET-CENTRE': 'CHATOU', 'LE VESINET-CEN': 'CHATOU',
    'LE VESINET-LE PECQ': 'CHATOU', 'LE VESINET-L.P': 'CHATOU',
    'LOGNES': 'TORCY',
    'NOGENT-SUR-MARNE': 'FONTENAY-SOUS-BOIS', 'NOGENT-S-MARNE': 'FONTENAY-SOUS-BOIS',
    'NOISIEL': 'TORCY', 'NOISY-CHAMPS': 'TORCY',
    "NOISY-LE-GRAND-MONT D'EST": 'NEUILLY-PLAISANCE', 'NOISY-LE-GRAND': 'NEUILLY-PLAISANCE',
    'RUEIL-MALMAISON': 'NANTERRE', 'RUEIL-MALMAIS.': 'NANTERRE',
    'SAINT-MAUR-CRETEIL': 'CHAMPIGNY', 'ST-MAUR-CRET.': 'CHAMPIGNY',
    'SUCY-BONNEUIL': 'CHAMPIGNY',
    "VAL-D'EUROPE": 'BUSSY-SAINT-GEORGES', "VAL D'EUROPE": 'BUSSY-SAINT-GEORGES',
    'VAL-DE-FONTENAY': 'FONTENAY-SOUS-BOIS', 'VAL D.FONTENAY': 'FONTENAY-SOUS-BOIS',
    'NEUVILLE UNIVERSITE': 'CERGY', 'NEUVILLE UNIVER': 'CERGY'
}

def trouver_ville_meteo(gare):
    if gare in gares_paris: return 'PARIS'
    if 'ST-GERMAIN' in gare: return 'SAINT-GERMAIN-EN-LAYE'
    for ville in villes_meteo_uniques:
        if ville in gare: return ville
    return fallbacks.get(gare, 'PARIS')

gares_uniques = df_trafic['LIBELLE_ARRET'].unique()
dictionnaire_mapping_rapide = {gare: trouver_ville_meteo(gare) for gare in gares_uniques}

# Application du dictionnaire pré-calculé
df_trafic['VILLE_CIBLE'] = df_trafic['LIBELLE_ARRET'].map(dictionnaire_mapping_rapide)


# 4. SÉCURITÉ MÉTÉO
print("🧹 Nettoyage des doublons météo...")
df_meteo = df_meteo.drop_duplicates(subset=['JOUR', 'VILLE_CIBLE'], keep='first')


# 5. FUSION FINALE
print("🔗 Croisement des bases de données...")
df_merged_final = pd.merge(df_trafic, df_meteo, on=['JOUR', 'VILLE_CIBLE'], how='left')

print("🩹 Comblement des trous générés par la disparition de certaines stations...")

# Inclusion de 'STATION_METEO' pour qu'elle se propage également
colonnes_meteo = ['STATION_METEO', 'METEO_GLOBALE', 'VENT_RAFALE_MS', 'PLUIE_MM', 'TEMP_MOYENNE_C']

# Étape A : Alignement par jour sur les autres gares d'Île-de-France actives
df_merged_final['JOUR'] = pd.to_datetime(df_merged_final['JOUR'])
df_merged_final[colonnes_meteo] = df_merged_final.groupby('JOUR')[colonnes_meteo].ffill().bfill()

# Étape B : Alignement chronologique par l'historique si coupure générale un jour donné
df_merged_final = df_merged_final.sort_values(by='JOUR')
df_merged_final[colonnes_meteo] = df_merged_final[colonnes_meteo].ffill().bfill()

# Remise du dataset dans l'ordre chronologique et structurel initial
df_merged_final = df_merged_final.sort_values(by=['JOUR', 'LIBELLE_ARRET', 'TRNC_HORR_60']).reset_index(drop=True)

# 6. SAUVEGARDE
df_merged_final.to_csv(chemin_datamart, index=False, sep=';', decimal=',')

print(f"\n🎉 SUCCÈS ABSOLU ! Fichier final est généré : {len(df_merged_final)} lignes parfaites.")
print("Aperçu des premières lignes avec la colonne station corrigée :")
print(df_merged_final[['JOUR', 'LIBELLE_ARRET', 'STATION_METEO', 'VILLE_CIBLE', 'METEO_GLOBALE', 'TEMP_MOYENNE_C']].head())

=== 🚀 SCRIPT 4 : LA FUSION FINALE (LE DATAMART) ===
📂 Chargement des fichiers...
🗺️ Attribution des gares RER à leurs zones Météo...
🧹 Nettoyage des doublons météo...
🔗 Croisement des bases de données...
🩹 Comblement des trous générés par la disparition de certaines stations...

🎉 SUCCÈS ABSOLU ! Fichier final est généré : 1219608 lignes parfaites.
Aperçu des premières lignes avec la colonne station corrigée :
        JOUR          LIBELLE_ARRET STATION_METEO VILLE_CIBLE METEO_GLOBALE  \
0 2015-01-01  ACHERES-GRAND-CORMIER       ACHERES     ACHERES   Temps Calme   
1 2015-01-01  ACHERES-GRAND-CORMIER       ACHERES     ACHERES   Temps Calme   
2 2015-01-01  ACHERES-GRAND-CORMIER       ACHERES     ACHERES   Temps Calme   
3 2015-01-01  ACHERES-GRAND-CORMIER       ACHERES     ACHERES   Temps Calme   
4 2015-01-01  ACHERES-GRAND-CORMIER       ACHERES     ACHERES   Temps Calme   

   TEMP_MOYENNE_C  
0             0.3  
1             0.3  
2             0.3  
3             0.3  
4          

## Persistance du fichier final



In [63]:
# Définition du chemin de sauvegarde
output_base_path = '/content/drive/MyDrive/YBOOST DATA B2/'

# Sauvegarde
df_merged_final.to_csv(os.path.join(output_base_path, 'RER_A_Final_FINAL.csv'), index=False)

print("Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.")

Fichier sauvegardé dans le dossier YBOOST DATA B2 ! Tu peux maintenant le charger comme un CSV classique.


## Vérifiactions et correction supplémentaire

In [64]:
df_merged_final.isnull().sum()

,0
PERIODE,0
JOUR,0
LIBELLE_ARRET,0
CAT_JOUR,0
NB_VALD_JOURNALIERE,0
TRNC_HORR_60,0
pourc_validations,0
NB_VALD_HORAIRE,0
VILLE_CIBLE,0
STATION_METEO,0


In [65]:
print(len(df_merged_final))

1219608


In [66]:
df_merged_final.head()

,PERIODE,JOUR,LIBELLE_ARRET,CAT_JOUR,NB_VALD_JOURNALIERE,TRNC_HORR_60,pourc_validations,NB_VALD_HORAIRE,VILLE_CIBLE,STATION_METEO,METEO_GLOBALE,VENT_RAFALE_MS,PLUIE_MM,TEMP_MOYENNE_C
0,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,0H-1H,0.09,1,ACHERES,ACHERES,Temps Calme,7.8,0.0,0.3
1,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,10H-11H,1.61,1,ACHERES,ACHERES,Temps Calme,7.8,0.0,0.3
2,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,11H-12H,1.76,1,ACHERES,ACHERES,Temps Calme,7.8,0.0,0.3
3,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,12H-13H,2.86,1,ACHERES,ACHERES,Temps Calme,7.8,0.0,0.3
4,2015S1,2015-01-01,ACHERES-GRAND-CORMIER,JOHV,21,13H-14H,9.06,1,ACHERES,ACHERES,Temps Calme,7.8,0.0,0.3


In [67]:
print("=== FINITIONS ET MISE EN FORME DU DATAMART ===")

# 1. Chemin unique du fichier final
chemin_fichier = '/content/drive/MyDrive/YBOOST DATA B2/RER_A_Final_FINAL.csv'

# 2. Chargement des données
print("📂 Chargement du fichier...")
df = pd.read_csv(chemin_fichier, sep=',', decimal='.')

# 3. TRADUCTION DE LA COLONNE CAT_JOUR
print("🔄 Humanisation des codes de jours...")
dictionnaire_types_jours = {
    'JOHV': 'Semaine (Hors Vacances)',
    'JOVR': 'Semaine (Vacances Scolaires)',
    'SAHV': 'Samedi (Hors Vacances)',
    'SAVR': 'Samedi (Vacances Scolaires)',
    'DIHV': 'Dimanche (Hors Vacances)',
    'DIVR': 'Dimanche (Vacances Scolaires)'
}

if 'CAT_JOUR' in df.columns:
    df = df.rename(columns={'CAT_JOUR': 'TYPE_JOUR'})
    df['TYPE_JOUR'] = df['TYPE_JOUR'].map(lambda x: dictionnaire_types_jours.get(x, x))

# 4. RÉORGANISATION DES COLONNES
print("📏 Application du nouvel ordre des colonnes...")
# L'ordre exact des colonnes (avec la modification de CAT_JOUR par TYPE_JOUR)
ordre_colonnes = [
    'PERIODE',
    'JOUR',
    'TYPE_JOUR',
    'LIBELLE_ARRET',
    'VILLE_CIBLE',
    'STATION_METEO',
    'NB_VALD_JOURNALIERE',
    'TRNC_HORR_60',
    'pourc_validations',
    'NB_VALD_HORAIRE',
    'TEMP_MOYENNE_C',
    'VENT_RAFALE_MS',
    'PLUIE_MM',
    'METEO_GLOBALE'
]

# On filtre pour garder cet ordre strict
colonnes_existantes = [col for col in ordre_colonnes if col in df.columns]
df = df[colonnes_existantes]

# 5. TRI DES LIGNES
print("⏳ Tri structurel du dataset (Jour > Arrêt > Heure)...")
df['JOUR'] = pd.to_datetime(df['JOUR'])
df = df.sort_values(by=['JOUR', 'LIBELLE_ARRET', 'TRNC_HORR_60']).reset_index(drop=True)

# 6. ÉCRASEMENT DU FICHIER ORIGINAL
print("💾 Sauvegarde en cours (Écrasement du fichier d'origine)...")
df.to_csv(chemin_fichier, index=False, sep=';', decimal=',')

print("\n🎉 Fichier mis à jour avec succès ! Dataset 100% prêt.")
print("🔍 Aperçu des premières lignes :")
print(df.head())

=== FINITIONS ET MISE EN FORME DU DATAMART ===
📂 Chargement du fichier...
🔄 Humanisation des codes de jours...
📏 Application du nouvel ordre des colonnes...
⏳ Tri structurel du dataset (Jour > Arrêt > Heure)...
💾 Sauvegarde en cours (Écrasement du fichier d'origine)...

🎉 Fichier mis à jour avec succès ! Dataset 100% prêt.
🔍 Aperçu des premières lignes :
  PERIODE       JOUR                TYPE_JOUR          LIBELLE_ARRET  \
0  2015S1 2015-01-01  Semaine (Hors Vacances)  ACHERES-GRAND-CORMIER   
1  2015S1 2015-01-01  Semaine (Hors Vacances)  ACHERES-GRAND-CORMIER   
2  2015S1 2015-01-01  Semaine (Hors Vacances)  ACHERES-GRAND-CORMIER   
3  2015S1 2015-01-01  Semaine (Hors Vacances)  ACHERES-GRAND-CORMIER   
4  2015S1 2015-01-01  Semaine (Hors Vacances)  ACHERES-GRAND-CORMIER   

  VILLE_CIBLE STATION_METEO  NB_VALD_JOURNALIERE TRNC_HORR_60  \
0     ACHERES       ACHERES                   21        0H-1H   
1     ACHERES       ACHERES                   21      10H-11H   
2     ACHERES  

In [68]:
print(len(df))

1219608


In [69]:
print("=== 🛠️ CORRECTION DES LES TYPES DE DONNÉES ===")

# 1. Listage toutes les colonnes qui doivent être des chiffres
colonnes_numeriques = [
    'NB_VALD_JOURNALIERE',
    'pourc_validations',
    'NB_VALD_HORAIRE',
    'TEMP_MOYENNE_C',
    'VENT_RAFALE_MS',
    'PLUIE_MM'
]

# 2. Transformations des nombres au format français en leur format iinternational ( "." au lieu de ",")
for col in colonnes_numeriques:
    df[col] = df[col].astype(str).str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

#3. Convertission en date (pour être sûr)
df['JOUR'] = pd.to_datetime(df['JOUR'])

print("✅ Typage terminé ! Voici les nouveaux types :")
print(df.info())

=== 🛠️ CORRECTION DES LES TYPES DE DONNÉES ===
✅ Typage terminé ! Voici les nouveaux types :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1219608 entries, 0 to 1219607
Data columns (total 14 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   PERIODE              1219608 non-null  object        
 1   JOUR                 1219608 non-null  datetime64[ns]
 2   TYPE_JOUR            1219608 non-null  object        
 3   LIBELLE_ARRET        1219608 non-null  object        
 4   VILLE_CIBLE          1219608 non-null  object        
 5   STATION_METEO        1219608 non-null  object        
 6   NB_VALD_JOURNALIERE  1219608 non-null  int64         
 7   TRNC_HORR_60         1219608 non-null  object        
 8   pourc_validations    1219608 non-null  float64       
 9   NB_VALD_HORAIRE      1219608 non-null  int64         
 10  TEMP_MOYENNE_C       1219608 non-null  float64       
 11  VENT_RAFALE_MS       1

# SAUVEGARDE FINALE
## on sauvegarde le fichier complet en format parquet pour pouvoir le stocker et l'exploiter plus facilement, tout en faisant un séquenssage par année du csv en plus petits csv pour pouvoir n'exploiter qu'une année spécifique et pouvoir les visualiser dans un logiciel (car le csv complet était trop lourd pour être visualisable dans excel)

In [70]:
print("=== 💾 EXPORT DOUBLE : PARQUET & CSV PAR ANNÉE ===")

# 1. DÉFINITION DE L'EMPLACEMENT ET DU NOM DU FICHIER
dossier_perso = '/content/drive/MyDrive/YBOOST DATA B2/Datamart_Exports/'
chemin_parquet = os.path.join(dossier_perso, 'RER_A_Final_FINAL.parquet')

# Création automatique du dossier sur le Drive s'il n'existe pas
if not os.path.exists(dossier_perso):
    os.makedirs(dossier_perso)
    print(f"📁 Dossier personnalisé créé : {dossier_perso}")
else:
    print(f"📁 Dossier cible détecté : {dossier_perso}")

# 2. SAUVEGARDE EN FORMAT PARQUET
print("\n📦 Sauvegarde du fichier complet en format Parquet...")
try:
    df_merged_final.to_parquet(chemin_parquet, index=False)
    print(f"✅ Succès ! Fichier Parquet compressé créé -> {chemin_parquet}")
except Exception as e:
    print(f"❌ Erreur lors de la sauvegarde Parquet : {e}")

# 3. SAUVEGARDE EN CSV PAR ANNÉE
print("\n✂️ Découpage du dataset et génération des CSV par année...")

df_merged_final['JOUR'] = pd.to_datetime(df_merged_final['JOUR'])

df_merged_final['ANNEE_PROVISOIRE'] = df_merged_final['JOUR'].dt.year

for annee, groupe in df_merged_final.groupby('ANNEE_PROVISOIRE'):
    nom_csv = f'RER_A_Final_{annee}.csv'
    chemin_csv_annee = os.path.join(dossier_perso, nom_csv)

    groupe_propre = groupe.drop(columns=['ANNEE_PROVISOIRE'])

    groupe_propre.to_csv(chemin_csv_annee, index=False, sep=';', decimal=',')
    print(f"💾 Fichier créé : {nom_csv} ({len(groupe_propre)} lignes)")

print("\n🎉 Opération terminée ! Tout est rangé proprement dans le dossier de destination.")

=== 💾 EXPORT DOUBLE : PARQUET & CSV PAR ANNÉE ===
📁 Dossier cible détecté : /content/drive/MyDrive/YBOOST DATA B2/Datamart_Exports/

📦 Sauvegarde du fichier complet en format Parquet...
✅ Succès ! Fichier Parquet compressé créé -> /content/drive/MyDrive/YBOOST DATA B2/Datamart_Exports/RER_A_Final_FINAL.parquet

✂️ Découpage du dataset et génération des CSV par année...
💾 Fichier créé : RER_A_Final_2015.csv (105648 lignes)
💾 Fichier créé : RER_A_Final_2016.csv (156168 lignes)
💾 Fichier créé : RER_A_Final_2017.csv (158880 lignes)
💾 Fichier créé : RER_A_Final_2018.csv (39312 lignes)
💾 Fichier créé : RER_A_Final_2019.csv (158256 lignes)
💾 Fichier créé : RER_A_Final_2020.csv (156336 lignes)
💾 Fichier créé : RER_A_Final_2021.csv (158976 lignes)
💾 Fichier créé : RER_A_Final_2022.csv (81168 lignes)
💾 Fichier créé : RER_A_Final_2023.csv (125664 lignes)
💾 Fichier créé : RER_A_Final_2024.csv (79200 lignes)

🎉 Opération terminée ! Tout est rangé proprement dans le dossier de destination.
